# Notebook 3 — Congestion Heatmap

**Technique:** Spatial binning (grid aggregation) — rasterising point data onto a regular grid

**Goal:** Produce a colour-coded congestion map of HCMC, browsable by hour of day.

**Method:**
1. Load all GPS events from MinIO historical archive
2. Snap each event to a 0.005° grid (~500m cells)
3. Compute mean speed per cell per hour
4. Render with Folium HeatMap layer
5. Compare hour 8 (morning rush) vs hour 14 (off-peak) vs April 30 hour 10 (parade)

In [ ]:
import boto3, json
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from botocore.client import Config
from datetime import datetime, timezone

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
BUCKET = 'bus-history'
CELL_SIZE = 0.005  # độ ≈ 500m tại TPHCM

In [ ]:
def load_events(max_objects=200):
    records = []
    paginator = s3.get_paginator('list_objects_v2')
    count = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix='2025-'):
        for obj in page.get('Contents', []):
            if count >= max_objects: break
            body = s3.get_object(Bucket=BUCKET, Key=obj['Key'])['Body'].read()
            records.extend(json.loads(body))
            count += 1
    return pd.DataFrame(records)

events = load_events(max_objects=300)
print(f'Loaded {len(events):,} events')

In [ ]:
events['x'] = pd.to_numeric(events['x'], errors='coerce')
events['y'] = pd.to_numeric(events['y'], errors='coerce')
events['speed'] = pd.to_numeric(events.get('speed'), errors='coerce')
events['datetime'] = pd.to_numeric(events['datetime'], errors='coerce')

# làm tròn về ô lưới
events['grid_lon'] = (events['x'] / CELL_SIZE).round() * CELL_SIZE
events['grid_lat'] = (events['y'] / CELL_SIZE).round() * CELL_SIZE

# tính giờ trong ngày
events['hour'] = events['datetime'].apply(
    lambda ts: datetime.fromtimestamp(ts, tz=timezone.utc).hour if pd.notna(ts) else None
)

# tổng hợp theo ô
grid = events.groupby(['grid_lat', 'grid_lon', 'hour'])['speed'].agg(
    mean_speed='mean', count='count'
).reset_index().dropna()
print(f'Grid cells: {len(grid):,}')

In [ ]:
def congestion_map(hour_filter):
    subset = grid[grid['hour'] == hour_filter]
    m = folium.Map(location=[10.7769, 106.7009], zoom_start=12, tiles='CartoDB dark_matter')

    # đảo tốc độ: chậm = tắc = màu nóng
    max_spd = subset['mean_speed'].max() or 1
    heat_data = [
        [row['grid_lat'], row['grid_lon'], max(0, 1.0 - row['mean_speed'] / max_spd)]
        for _, row in subset.iterrows()
    ]
    HeatMap(heat_data, radius=18, blur=12, min_opacity=0.3).add_to(m)
    folium.map.Marker(
        [11.1, 106.4],
        icon=folium.DivIcon(html=f'<div style="color:white;font-size:14px">Hour {hour_filter}:00 ICT</div>')
    ).add_to(m)
    return m

# giờ cao điểm sáng
m8 = congestion_map(8)
m8.save('/tmp/congestion_hour08.html')
print('Morning rush map saved.')
m8

In [ ]:
m14 = congestion_map(14)
m14.save('/tmp/congestion_hour14.html')
print('Off-peak map saved.')
m14